In [ ]:
# Mounting the progress to google drive
# Rerun this cell everytime
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# Enabling gpu and verifying
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
# Installing required libraries

!pip install -q \
    transformers \
    datasets \
    accelerate>=0.26.0 \
    scikit-learn \
    pandas \
    pyarrow


In [ ]:
# Required imports

import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Install faiss
!pip install -q faiss-cpu sentence-transformers transformers accelerate protobuf scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 95.1 MB/s eta 0:00:00


In [ ]:
!pip install -q wikidata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
!pip install -q spacy wikipedia-api
!python -m spacy download en_core_web_sm


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 128.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Entity extraction + Wikipedia linking
import spacy
import wikipediaapi

# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Initialize Wikipedia API (user_agent REQUIRED)
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="FakeNewsDetection/1.0 (academic project)"
)




In [ ]:
# Loading all our progress

# 2. Load classifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = "/content/drive/MyDrive/fake_news_system/classifier"

tokenizer_clf = AutoTokenizer.from_pretrained(MODEL_PATH)
model_clf = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
model_clf.eval()

# 3. Load FAISS + corpus
import faiss, pickle

FAISS_DIR = "/content/drive/MyDrive/fake_news_system/faiss"

index = faiss.read_index(f"{FAISS_DIR}/index.faiss")

with open(f"{FAISS_DIR}/corpus_texts.pkl", "rb") as f:
    corpus_texts = pickle.load(f)

# 4. Load retriever
from sentence_transformers import SentenceTransformer
retriever = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

# 5. Load NLI model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
nli_tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")
nli_model = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli").to(DEVICE)
nli_model.eval()

The tokenizer you are loading from '/content/drive/MyDrive/fake_news_system/classifier' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

## Evidence aware fusion

In [ ]:
def aggregate_nli_weighted(nli_results):
    weighted_entail = 0.0
    weighted_contra = 0.0
    weighted_neutral = 0.0

    for r in nli_results:
        w = r["similarity"]

        weighted_entail += w * r["entailment"]
        weighted_contra += w * r["contradiction"]
        weighted_neutral += w * r["neutral"]

    total = weighted_entail + weighted_contra + weighted_neutral + 1e-8

    return {
        "entail_score": weighted_entail / total,
        "contra_score": weighted_contra / total,
        "neutral_score": weighted_neutral / total
    }





def evidence_aware_prediction_weighted(
    classifier_prob,
    nli_scores,
    entail_thresh=0.6,
    contra_thresh=0.6
):
    entail = nli_scores["entail_score"]
    contra = nli_scores["contra_score"]

    if contra > contra_thresh:
        return {
            "label": "FAKE",
            "reason": "High-confidence contradictory evidence",
            "confidence": contra
        }

    if entail > entail_thresh:
        return {
            "label": "REAL",
            "reason": "High-confidence supporting evidence",
            "confidence": entail
        }

    label = "REAL" if classifier_prob >= 0.5 else "FAKE"
    return {
        "label": label,
        "reason": "Classifier-based decision (weighted evidence inconclusive)",
        "confidence": classifier_prob
    }





In [ ]:
# Classifier inference helper
import torch
import numpy as np

def classifier_predict(claim, tokenizer, model, device):
    """
    Returns probability of REAL (label=1)
    """
    inputs = tokenizer(
        claim,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # assuming label mapping: 0=FAKE, 1=REAL
    return float(probs[1]), probs


In [ ]:
# After helper functions, before retrieval logic
def extract_entities(text):
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "EVENT", "GPE", "WORK_OF_ART"}:
            entities.append(ent.text)
    return list(set(entities))


def link_entity_to_wikipedia(entity):
    page = wiki.page(entity)
    if page.exists():
        return page.title
    return None


In [ ]:
def retrieve_evidence(
    claim,
    retriever_model,
    faiss_index,
    corpus_texts,
    top_k=5
):
    claim_emb = retriever_model.encode(
        [claim],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    D, I = faiss_index.search(claim_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            "text": corpus_texts[idx],
            "similarity": float(score)
        })

    return results


In [ ]:
# Q-id resolver from wikipedia page
from wikidata.client import Client

wikidata_client = Client()

In [ ]:
def get_wikidata_qid(entity_name):
    """
    Resolve entity → Wikidata Q-ID using Wikipedia + Wikidata
    """
    page = wiki.page(entity_name)
    if not page.exists():
        return None

    try:
        wd_item = wikidata_client.get(page.wikidata_id, load=True)
        return wd_item.id  # e.g. Q11571
    except Exception:
        return None


In [ ]:
def extract_entities_with_qids(text):
    doc = nlp(text)
    entities = {}

    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "EVENT", "GPE", "WORK_OF_ART"}:
            qid = get_wikidata_qid(ent.text)
            if qid:
                entities[ent.text] = qid

    return entities



In [ ]:
def filter_evidence_by_qid(claim_entities, retrieved_results):
    """
    Keep only evidence that shares at least one Wikidata Q-ID with claim
    """
    if not claim_entities:
        return retrieved_results  # fallback

    claim_qids = set(claim_entities.values())
    filtered = []

    for r in retrieved_results:
        evidence_entities = extract_entities_with_qids(r["text"])
        evidence_qids = set(evidence_entities.values())

        if claim_qids & evidence_qids:
            filtered.append(r)

    return filtered


In [ ]:
# Filter evidence by  Q-id
def filter_evidence(claim, retrieved_results):
    # Step 1: Try Wikidata Q-ID filtering
    claim_entities_qid = extract_entities_with_qids(claim)
    filtered = filter_evidence_by_qid(claim_entities_qid, retrieved_results)

    if len(filtered) >= 2:
        return filtered

    # Step 2: Fallback to entity-level filtering
    filtered = filter_evidence_by_entity(claim, retrieved_results)

    if filtered:
        return filtered

    # Step 3: Last fallback — return raw retrieval
    return retrieved_results


In [ ]:
# Immediately AFTER retrieve_evidence() function
def filter_evidence_by_entity(claim, retrieved_results):
    claim_entities = extract_entities(claim)

    claim_links = {
        link_entity_to_wikipedia(ent)
        for ent in claim_entities
        if link_entity_to_wikipedia(ent) is not None
    }

    if not claim_links:
        return retrieved_results  # fallback if no entity detected

    filtered = []
    for r in retrieved_results:
        text_entities = extract_entities(r["text"])
        text_links = {
            link_entity_to_wikipedia(ent)
            for ent in text_entities
            if link_entity_to_wikipedia(ent) is not None
        }

        if claim_links & text_links:
            filtered.append(r)

    return filtered


In [ ]:
# -----------------------------
# Entity-Type Fallback Helpers
# -----------------------------

PROFESSION_KEYWORDS = {
    "cricketer", "footballer", "player", "athlete",
    "actor", "actress", "politician", "singer",
    "scientist", "author", "writer", "director",
    "businessman", "entrepreneur"
}

def is_profession_claim(text: str) -> bool:
    text_lower = text.lower()
    return any(prof in text_lower for prof in PROFESSION_KEYWORDS)


def has_real_world_entity(claim: str) -> bool:
    entities = extract_entities(claim)
    for ent in entities:
        if link_entity_to_wikipedia(ent) is not None:
            return True
    return False


In [ ]:
import re

def extract_entity(claim: str):
    """
    Extracts the main entity from simple factual claims.
    Example:
      'Virat Kohli is a cricketer.' -> 'virat kohli'
    """
    claim = claim.lower()

    patterns = [
        r"(.+?) is ",
        r"(.+?) was ",
        r"(.+?) are ",
        r"(.+?) plays ",
        r"(.+?) played "
    ]

    for p in patterns:
        m = re.match(p, claim)
        if m:
            return m.group(1).strip()

    # fallback: first 3 words
    return " ".join(claim.split()[:3])


In [ ]:
def entity_aware_filter(claim, retrieved_evidence):
    """
    Keeps only evidence that mentions the same entity as the claim.
    """
    entity = extract_entity(claim)

    filtered = [
        r for r in retrieved_evidence
        if entity in r["text"].lower()
    ]

    return filtered


In [ ]:
def safe_evidence_selection(claim, retrieved_evidence, min_required=1):
    filtered = entity_aware_filter(claim, retrieved_evidence)

    if len(filtered) < min_required:
        return None  # tells system: evidence unreliable

    return filtered


In [ ]:
def nli_verify_evidence(claim, retrieved_results):
    results = []

    for r in retrieved_results:
        premise = r["text"]
        similarity = r["similarity"]

        inputs = nli_tokenizer(
            premise,
            claim,
            return_tensors="pt",
            truncation=True
        ).to(DEVICE)

        with torch.no_grad():
            logits = nli_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[0]

        results.append({
            "evidence": premise,
            "similarity": similarity,
            "entailment": probs[2].item(),
            "contradiction": probs[0].item(),
            "neutral": probs[1].item()
        })

    return results


In [ ]:
# Evidence Reliability Check
def is_evidence_reliable(
    retrieved_results,
    nli_scores,
    min_similarity=0.6,
    min_entail_or_contra=0.7
):
    """
    Determines whether retrieved evidence is strong enough
    to override classifier.
    """

    if not retrieved_results:
        return False

    # Check similarity strength
    strong_sim = any(
        r["similarity"] >= min_similarity
        for r in retrieved_results
    )

    # Check NLI confidence
    strong_nli = (
        nli_scores["entail_score"] >= min_entail_or_contra
        or nli_scores["contra_score"] >= min_entail_or_contra
    )

    return strong_sim and strong_nli


In [ ]:
def evidence_aware_prediction_with_unverifiable(
    classifier_prob,
    nli_scores,
    retrieved_results,
    entail_thresh=0.7,
    contra_thresh=0.7
):
    """
    classifier_prob: probability of REAL from classifier
    nli_scores: output of aggregate_nli_weighted()
    retrieved_results: FAISS retrieval output
    """

    entail = nli_scores["entail_score"]
    contra = nli_scores["contra_score"]

    # 1️⃣ Strong contradiction → FAKE
    if contra >= contra_thresh:
        return {
            "label": "FAKE",
            "reason": "High-confidence contradictory evidence",
            "confidence": contra
        }

    # 2️⃣ Strong entailment → REAL
    if entail >= entail_thresh:
        return {
            "label": "REAL",
            "reason": "High-confidence supporting evidence",
            "confidence": entail
        }

    # 3️⃣ Evidence reliability check
    reliable = is_evidence_reliable(
        retrieved_results,
        nli_scores
    )

    if not reliable:
        return {
            "label": "UNVERIFIABLE",
            "reason": "No credible real-world evidence found",
            "confidence": 1 - classifier_prob
        }

    # 4️⃣ Weak evidence → classifier fallback (rare now)
    label = "REAL" if classifier_prob >= 0.5 else "FAKE"
    return {
        "label": label,
        "reason": "Weak evidence; classifier fallback",
        "confidence": classifier_prob
    }


def evidence_dominance(classifier_prob, nli_scores, margin=0.1):

    entail = nli_scores["entail_score"]
    contra = nli_scores["contra_score"]

    # 🚨 Override contradictory noise
    if classifier_prob >= 0.9 and entail >= 0.25:
        return {
            "label": "REAL",
            "reason": "High-confidence classifier with supporting evidence",
            "confidence": classifier_prob
        }

    if entail - contra >= margin:
        return {
            "label": "REAL",
            "reason": "Evidence majority supports the claim",
            "confidence": entail
        }

    if contra - entail >= margin:
        return {
            "label": "FAKE",
            "reason": "Evidence majority contradicts the claim",
            "confidence": contra
        }

    return {
        "label": "REAL" if classifier_prob >= 0.5 else "FAKE",
        "reason": "Mixed evidence; trusted classifier",
        "confidence": classifier_prob
    }


def is_negated_variant(text1, text2):
    """
    Heuristic: checks if texts differ mainly by negation
    """
    negations = {"not", "does not", "do not", "no", "never"}

    t1 = text1.lower()
    t2 = text2.lower()

    for neg in negations:
        if neg in t1 and neg not in t2:
            return True
        if neg in t2 and neg not in t1:
            return True

    return False

def collapse_negated_evidence(retrieved_results):
    cleaned = []

    for r in retrieved_results:
        is_duplicate = False
        for kept in cleaned:
            if is_negated_variant(r["text"], kept["text"]):
                # Keep the one with higher similarity
                if r["similarity"] > kept["similarity"]:
                    kept["text"] = r["text"]
                    kept["similarity"] = r["similarity"]
                is_duplicate = True
                break

        if not is_duplicate:
            cleaned.append(r)

    return cleaned


def is_verifiable_claim(claim):
    """
    Returns False for speculative / extraordinary claims
    """
    speculative_keywords = [
        "alien", "ufo", "invasion", "attacked the earth",
        "time travel", "parallel universe",
        "immortal", "teleport", "conspiracy"
    ]

    claim_lower = claim.lower()
    for kw in speculative_keywords:
        if kw in claim_lower:
            return False

    return True






## We add Similarity-Weighted Voting to our system
#### Weight each NLI score by how similar the evidence is to the claim.

##### Highly similar evidence = Strong vote
##### Low similarity evidence = Weak vote

##### We do this so that not every extracted evidence is given equal importance

In [ ]:
import json

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


In [ ]:
pib_docs = load_jsonl(
    "/content/drive/MyDrive/fake_news_data/pib_2025_corpus_12.jsonl"
)

sc_docs = load_jsonl(
    "/content/drive/MyDrive/fake_news_data/supreme_court_2025_1.jsonl"
)

print("PIB docs:", len(pib_docs))
print("SC docs:", len(sc_docs))


PIB docs: 1462
SC docs: 388


In [ ]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


### Load FAISS Indexes

In [ ]:
import faiss

faiss_pib = faiss.read_index(
    "/content/drive/MyDrive/fake_news_system/faiss_pib.index"
)

faiss_sc = faiss.read_index(
    "/content/drive/MyDrive/fake_news_system/faiss_sc.index"
)

print("PIB FAISS size:", faiss_pib.ntotal)
print("SC FAISS size:", faiss_sc.ntotal)


PIB FAISS size: 3597
SC FAISS size: 15463


In [ ]:
import pickle

with open("/content/drive/MyDrive/fake_news_system/pib_corpus.pkl", "rb") as f:
    pib_corpus = pickle.load(f)

with open("/content/drive/MyDrive/fake_news_system/sc_corpus.pkl", "rb") as f:
    sc_corpus = pickle.load(f)

print("PIB corpus size:", len(pib_corpus))
print("SC corpus size:", len(sc_corpus))


PIB corpus size: 3597
SC corpus size: 15463


In [ ]:
# Already saved earlier — just load
import faiss, json

faiss_rbi = faiss.read_index(
    "/content/drive/MyDrive/fake_news_data/faiss_rbi.index"
)

with open("/content/drive/MyDrive/fake_news_data/rbi_corpus_2025.json") as f:
    rbi_corpus = json.load(f)

print("RBI corpus:", len(rbi_corpus))
print("RBI FAISS:", faiss_rbi.ntotal)


RBI corpus: 404
RBI FAISS: 404


In [ ]:
def is_pre_announcement_policy_claim(claim):
    claim = claim.lower()
    indicators = [
        "planning to",
        "reportedly planning",
        "internal approval",
        "pilot program",
        "sources claim",
        "officials familiar",
        "no public notification",
        "no tender",
        "yet to be announced",
        "expected to be rolled out"
    ]
    return any(i in claim for i in indicators)


In [ ]:
def is_policy_specific_match(claim, text):
    claim = claim.lower()
    text = text.lower()

    # Extract key policy intents from claim
    critical_terms = [
        "abolish", "scrap", "remove", "eliminate",
        "income tax", "direct tax",
        "replace income tax",
        "gst expansion"
    ]

    # If claim talks about these, evidence MUST also mention them
    required = [t for t in critical_terms if t in claim]

    if not required:
        return True  # fallback for non-specific claims

    return all(t in text for t in required)


In [ ]:
def is_clarificatory_policy_claim(claim):
    cues = [
        "reiterated",
        "clarified",
        "no expansion",
        "existing legal provisions",
        "judicial directions",
        "subject to law",
        "no change approved",
        "within the current framework"
    ]
    c = claim.lower()
    return any(k in c for k in cues)


In [ ]:
# Add authoritative gate
# 1. Claim routing
def route_claim(claim):
    c = claim.lower()

    # -----------------------------
    # RBI (Monetary Authority) — MUST BE FIRST
    # -----------------------------
    if any(k in c for k in [
        "reserve bank of india",
        "rbi",
        "monetary policy",
        "repo rate",
        "reverse repo",
        "interest rate",
        "negative interest",
        "savings account",
        "deposit rate"
    ]):
        return "RBI"

    # -----------------------------
    # Supreme Court (Judiciary)
    # -----------------------------
    if any(k in c for k in [
        "supreme court",
        "judgment",
        "verdict",
        "bench",
        "civil appeal",
        "criminal appeal",
        "article 32",
        "article 226"
    ]):
        return "SC"

    # -----------------------------
    # Government / PIB
    # -----------------------------
    if any(k in c for k in [
        "government",
        "ministry",
        "policy",
        "scheme",
        "income tax",
        "gst",
        "budget"
    ]):
        return "PIB"

    return "GENERAL"






In [ ]:
def is_authoritative_interpretive_claim(claim):
    c = claim.lower()

    interpretive_cues = [
        # explicit interpretation language
        "clarifying the interpretation",
        "interpretation of",
        "interpreting",
        "clarifies",
        "clarified",

        # judicial reasoning language
        "the court observed",
        "the bench observed",
        "the bench stressed",
        "the court emphasized",
        "the judgment explains",
        "the ruling explains",
        "the ruling reinforces",

        # meta-judicial language
        "legal experts have noted",
        "expected to guide",
        "guidance to lower courts",
        "judicial discipline",
        "procedural fairness",
        "balancing procedural",
        "consistent handling",
        "reinforces principles",

        # NOT outcome language
        "emphasizing that",
        "stressed that"
    ]

    return any(cue in c for cue in interpretive_cues)


In [ ]:
def is_sc_ruling_specific(text, claim):
    text = text.lower()
    claim = claim.lower()

    # 1️⃣ Must be a Supreme Court ruling
    ruling_indicators = [
        "supreme court held",
        "supreme court ruled",
        "supreme court ordered",
        "the court held",
        "the court ruled",
        "judgment",
        "verdict"
    ]

    is_ruling = any(ind in text for ind in ruling_indicators)
    if not is_ruling:
        return False

    # 2️⃣ Must match claim topic
    if "upi" in claim or "digital payment" in claim:
        return any(t in text for t in [
            "upi", "digital payment", "npci", "payment system"
        ])

    # fallback for other SC claims
    return True


In [ ]:
def is_rbi_policy_specific(claim, text):
    claim = claim.lower()
    text = text.lower()

    policy_terms = [
        "negative interest",
        "repo rate",
        "reverse repo",
        "monetary policy",
        "liquidity",
        "savings account",
        "deposit rate"
    ]

    required = [t for t in policy_terms if t in claim]
    if not required:
        return True

    return any(t in text for t in required)


In [ ]:
def is_clarificatory_policy_claim(claim):
    cues = [
        "clarified", "reiterated", "no expansion",
        "existing legal provisions", "within the current framework",
        "no change approved"
    ]
    return any(k in claim.lower() for k in cues)

In [ ]:
# 2. Authoritative Retrieval
def retrieve_from_authority(claim, index, corpus, top_k=5, min_sim=0.5):
    query_emb = retriever.encode(
        [claim],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    D, I = index.search(query_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        if score < min_sim:
            continue

        doc = corpus[idx]
        results.append({
            "text": doc["text"],
            "similarity": float(score),
            "authority": doc["authority"],
            "source": doc["source"],
            "date": doc["date"]
        })

    return results

In [ ]:
def authoritative_gate(claim):
    route = route_claim(claim)

    # -----------------------------
    # RBI MONETARY POLICY GATE
    # -----------------------------
    if route == "RBI":
        evidence = retrieve_from_authority(
            claim, faiss_rbi, rbi_corpus
        )

        rbi_confirming = [
            e for e in evidence
            if is_rbi_policy_specific(claim, e["text"])
        ]

        if not rbi_confirming:
            return {
                "block": True,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "No official RBI confirmation for this specific monetary policy claim",
                    "confidence": 1.0
                }
            }

        return {"block": False,
            "clarificatory": False,
            "interpretive": False}

    # -----------------------------
    # PIB GOVERNMENT POLICY GATE
    # -----------------------------
    if route == "PIB":
        evidence = retrieve_from_authority(
            claim, faiss_pib, pib_corpus
        )

        # 🚨 Hard gate for sensitive surveillance claims
        if (
            any(k in claim.lower() for k in [
                "aadhaar",
                "facial recognition",
                "surveillance",
                "ai-based monitoring",
                "real-time identification"
            ])
            and is_pre_announcement_policy_claim(claim)
        ):
            return {
                "block": True,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "Sensitive government surveillance policy lacks official public notification",
                    "confidence": 1.0
                }
            }

        policy_confirming = [
            e for e in evidence
            if is_policy_specific_match(claim, e["text"])
        ]

        if not policy_confirming:
            # ✅ Allow clarificatory statements to proceed
            if is_clarificatory_policy_claim(claim):
                return {"block": False, "clarificatory": True}

            return {
                "block": True,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "No official PIB confirmation for this specific government policy claim",
                    "confidence": 1.0
                }
            }

        return {"block": False,
            "clarificatory": False,
            "interpretive": False}

    # -----------------------------
    # SUPREME COURT LEGAL GATE
    # -----------------------------
    if route == "SC":
        print("DEBUG SC INTERPRETIVE CHECK:", is_authoritative_interpretive_claim(claim))
        evidence = retrieve_from_authority(
            claim, faiss_sc, sc_corpus
        )

        ruling_confirming = [
            e for e in evidence
            if is_sc_ruling_specific(e["text"], claim)
        ]

        if not ruling_confirming:
            if is_authoritative_interpretive_claim(claim):
                return {
                    "block": False,
                    "interpretive": True,
                    "clarificatory": False
                }

            return {
                "block": True,
                "final_prediction": {
                    "label": "FAKE",
                    "reason": "No Supreme Court ruling or judgment confirms this claim",
                    "confidence": 1.0
                }
            }

        return {
            "block": False,
            "clarificatory": False,
            "interpretive": False
        }



    # -----------------------------
    # GENERAL CLAIMS
    # -----------------------------
    return {"block": False,
            "clarificatory": False,
            "interpretive": False}


In [ ]:
def predict_with_evidence(
    claim,
    clf_tokenizer,
    clf_model,
    retriever,
    faiss_index,
    corpus_texts,
    nli_tokenizer,
    nli_model,
    device,
    top_k=5
):
    # -----------------------------
    # 0️⃣ Authoritative Gate
    # -----------------------------
    gate = authoritative_gate(claim)

    if gate.get("block"):
        return {
            "claim": claim,
            "final_prediction": gate["final_prediction"],
            "retrieved_evidence": [],
            "nli_scores": None,
            "note": "Authoritative hard gate applied"
        }

    clarificatory = gate.get("clarificatory", False)
    interpretive = gate.get("interpretive", False)
    print("DEBUG GATE:", gate)
    print("DEBUG INTERPRETIVE:", interpretive)


    # -----------------------------
    # 1️⃣ Classifier prediction
    # -----------------------------
    clf_prob_real, clf_probs = classifier_predict(
        claim, clf_tokenizer, clf_model, device
    )

    # -----------------------------
    # 2️⃣ Retrieve evidence
    # -----------------------------
    retrieved_raw = retrieve_evidence(
        claim,
        retriever,
        faiss_index,
        corpus_texts,
        top_k=top_k
    )

    retrieved = retrieved_raw.copy()

    # -----------------------------
    # 3️⃣ Evidence cleaning
    # -----------------------------
    retrieved_dedup = collapse_negated_evidence(retrieved)
    retrieved_entity = filter_evidence(claim, retrieved_dedup)
    filtered_evidence = safe_evidence_selection(claim, retrieved_entity)

    # -----------------------------
    # 4️⃣ TRUE UNVERIFIABLE LOGIC
    # -----------------------------
    if not filtered_evidence:

        if not has_real_world_entity(claim):
            return {
                "claim": claim,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "No credible real-world evidence found",
                    "confidence": 1.0 - clf_prob_real
                },
                "retrieved_evidence": retrieved,
                "nli_scores": None
            }

        if (
            is_profession_claim(claim)
            and clf_prob_real >= 0.7
            and route_claim(claim) == "GENERAL"
        ):
            return {
                "claim": claim,
                "final_prediction": {
                    "label": "REAL",
                    "reason": "Well-known entity profession; classifier trusted",
                    "confidence": clf_prob_real
                },
                "retrieved_evidence": retrieved,
                "nli_scores": None
            }

        if not is_verifiable_claim(claim):
            return {
                "claim": claim,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "Claim is speculative or unverifiable",
                    "confidence": 1.0 - clf_prob_real
                },
                "retrieved_evidence": retrieved,
                "nli_scores": None
            }

        # ❌ Block classifier-only decisions for authoritative claims
        route = route_claim(claim)
        if route in {"PIB", "RBI", "SC"} and not (clarificatory or interpretive):
            return {
                "claim": claim,
                "final_prediction": {
                    "label": "UNVERIFIABLE",
                    "reason": "Authoritative claim without sufficient official confirmation",
                    "confidence": 1.0
                },
                "retrieved_evidence": retrieved,
                "nli_scores": None
            }

        # ✅ Allowed fallback
        return {
            "claim": claim,
            "final_prediction": {
                "label": "REAL" if clf_prob_real >= 0.5 else "FAKE",
                "reason": (
                    "Clarificatory or interpretive authoritative statement"
                    if (clarificatory or interpretive)
                    else "Verifiable claim with no strong evidence; classifier trusted"
                ),
                "confidence": clf_prob_real
            },
            "retrieved_evidence": retrieved,
            "nli_scores": None
        }

    # -----------------------------
    # 5️⃣ NLI verification
    # -----------------------------
    nli_results = nli_verify_evidence(claim, filtered_evidence)

    # -----------------------------
    # 6️⃣ Aggregate NLI
    # -----------------------------
    nli_scores = aggregate_nli_weighted(nli_results)

    # -----------------------------
    # 7️⃣ Evidence dominance
    # -----------------------------
    final_decision = evidence_dominance(
        classifier_prob=clf_prob_real,
        nli_scores=nli_scores
    )

    # -----------------------------
    # 8️⃣ Final output
    # -----------------------------
    return {
        "claim": claim,
        "final_prediction": final_decision,
        "classifier_probs": clf_probs.tolist(),
        "nli_scores": nli_scores,
        "retrieved_evidence": filtered_evidence,
        "nli_details": nli_results
    }


In [ ]:
# Example Usage
claim = """ The Supreme Court has delivered a detailed judgment clarifying the interpretation of Section 5 of the Limitation Act, emphasizing that courts must carefully assess whether sufficient cause exists when condoning delays in filing appeals. The ruling arose from a long-pending civil dispute involving land ownership and compensation issues.

In its judgment, the court observed that while procedural delays involving government bodies may sometimes be unavoidable due to administrative processes, such delays cannot be condoned as a matter of routine. The bench stressed that public authorities are expected to act with diligence and accountability, and excessive delay without justification undermines the rule of law.

Legal experts have noted that the ruling reinforces judicial discipline while balancing procedural fairness. The judgment is expected to guide lower courts in handling delay condonation applications more consistently."""



output = predict_with_evidence(
    claim=claim,
    clf_tokenizer=tokenizer_clf,
    clf_model=model_clf,
    retriever=retriever,
    faiss_index=index,
    corpus_texts=corpus_texts,
    nli_tokenizer=nli_tokenizer,
    nli_model=nli_model,
    device=DEVICE
)

print("🔮 FINAL DECISION")
print(output["final_prediction"])

print("\n🧠 NLI SCORES")
if "nli_scores" in output:
    print("\n🧠 NLI SCORES")
    print(output["nli_scores"])
else:
    print("\n🧠 NLI SCORES")
    print("Skipped (no reliable evidence)")


print("\n📚 TOP EVIDENCE")
for ev in output["retrieved_evidence"]:
    print("-", ev["text"])




DEBUG SC INTERPRETIVE CHECK: True
DEBUG GATE: {'block': False, 'clarificatory': False, 'interpretive': False}
DEBUG INTERPRETIVE: False
🔮 FINAL DECISION
{'label': 'UNVERIFIABLE', 'reason': 'Authoritative claim without sufficient official confirmation', 'confidence': 1.0}

🧠 NLI SCORES

🧠 NLI SCORES
None

📚 TOP EVIDENCE
- The Magna Carta pledged to give access to timely justice.
- Richmond, Virginia is home to one of the courts of appeals.
- The Pelican Brief is a 1993 United States Supreme Court decision.
- Constitutional instruction can be involved with a loss of supply.
- Above the Law is an action company.


In [ ]:
print(authoritative_gate(claim))


DEBUG SC INTERPRETIVE CHECK: True
{'block': False, 'clarificatory': False, 'interpretive': False}


In [ ]:
import inspect
print(inspect.getsource(is_authoritative_interpretive_claim))


def is_authoritative_interpretive_claim(claim):
    c = claim.lower()

    interpretive_cues = [
        # explicit interpretation language
        "clarifying the interpretation",
        "interpretation of",
        "interpreting",
        "clarifies",
        "clarified",

        # judicial reasoning language
        "the court observed",
        "the bench observed",
        "the bench stressed",
        "the court emphasized",
        "the judgment explains",
        "the ruling explains",
        "the ruling reinforces",

        # meta-judicial language
        "legal experts have noted",
        "expected to guide",
        "guidance to lower courts",
        "judicial discipline",
        "procedural fairness",
        "balancing procedural",
        "consistent handling",
        "reinforces principles",

        # NOT outcome language
        "emphasizing that",
        "stressed that"
    ]

    return any(cue in c for cue in interpretive_cues)



In [ ]:
# Example Usage
claim = """ The Reserve Bank of India has decided to maintain the repo rate at its existing level, citing concerns over global economic volatility and domestic inflation trends. The decision was announced following the latest meeting of the Monetary Policy Committee, which reviewed growth prospects and price stability indicators.

The central bank noted that while inflation has moderated in recent months, risks remain due to geopolitical tensions and fluctuations in commodity prices. RBI officials reiterated their commitment to supporting economic growth while ensuring inflation remains within the target range.

Market participants largely welcomed the decision, stating that policy stability would provide clarity to borrowers and investors. The RBI also signaled that future policy actions would remain data-dependent. """



output = predict_with_evidence(
    claim=claim,
    clf_tokenizer=tokenizer_clf,
    clf_model=model_clf,
    retriever=retriever,
    faiss_index=index,
    corpus_texts=corpus_texts,
    nli_tokenizer=nli_tokenizer,
    nli_model=nli_model,
    device=DEVICE
)

print("🔮 FINAL DECISION")
print(output["final_prediction"])

print("\n🧠 NLI SCORES")
if "nli_scores" in output:
    print("\n🧠 NLI SCORES")
    print(output["nli_scores"])
else:
    print("\n🧠 NLI SCORES")
    print("Skipped (no reliable evidence)")


print("\n📚 TOP EVIDENCE")
for ev in output["retrieved_evidence"]:
    print("-", ev["text"])




DEBUG GATE: {'block': False, 'clarificatory': False, 'interpretive': False}
DEBUG INTERPRETIVE: False
🔮 FINAL DECISION
{'label': 'UNVERIFIABLE', 'reason': 'Authoritative claim without sufficient official confirmation', 'confidence': 1.0}

🧠 NLI SCORES

🧠 NLI SCORES
None

📚 TOP EVIDENCE
- Global Finance is included in International Relations.
- India has a government.
- The World Bank Group's activities are focused on developing countries.
- The Philippines's market is emerging.
- The World Bank Group's activities include protection of the environment.


In [ ]:
# Example Usage
claim = """ The Ministry of Railways has launched a new digital passenger information system across several major railway stations to improve real-time travel updates and crowd management. The initiative includes upgraded display boards, automated announcements, and integration with existing railway information platforms.

Officials stated that the system is designed to enhance passenger convenience, reduce congestion, and improve overall travel experience. The rollout follows successful pilot implementations at select stations earlier this year.

The ministry added that further expansion will be undertaken in phases, with feedback from passengers and station authorities guiding future enhancements. """



output = predict_with_evidence(
    claim=claim,
    clf_tokenizer=tokenizer_clf,
    clf_model=model_clf,
    retriever=retriever,
    faiss_index=index,
    corpus_texts=corpus_texts,
    nli_tokenizer=nli_tokenizer,
    nli_model=nli_model,
    device=DEVICE
)

print("🔮 FINAL DECISION")
print(output["final_prediction"])

print("\n🧠 NLI SCORES")
if "nli_scores" in output:
    print("\n🧠 NLI SCORES")
    print(output["nli_scores"])
else:
    print("\n🧠 NLI SCORES")
    print("Skipped (no reliable evidence)")


print("\n📚 TOP EVIDENCE")
for ev in output["retrieved_evidence"]:
    print("-", ev["text"])




DEBUG GATE: {'block': False, 'clarificatory': False, 'interpretive': False}
DEBUG INTERPRETIVE: False
🔮 FINAL DECISION
{'label': 'UNVERIFIABLE', 'reason': 'No credible real-world evidence found', 'confidence': 0.07292866706848145}

🧠 NLI SCORES

🧠 NLI SCORES
None

📚 TOP EVIDENCE
- Event management includes transportation.
- Passengers was released in 2017.
- Passengers is a television series.
- Event management includes coordinating transportation.
- Passengers is a recorded work.


In [ ]:
# Example Usage
claim = """ The government has reiterated that Aadhaar usage will continue to be governed strictly by existing legal provisions and judicial directions. In a statement issued in Parliament, officials clarified that Aadhaar authentication is permitted only for purposes explicitly allowed under law and subject to data protection safeguards.

The clarification follows public discussions around digital identity and privacy concerns. Authorities emphasized that no expansion of Aadhaar usage beyond the current framework has been approved, and any future changes would require legislative and judicial scrutiny.

Privacy advocates welcomed the clarification, stating that transparency and legal oversight are essential to maintaining public trust in digital governance systems. """



output = predict_with_evidence(
    claim=claim,
    clf_tokenizer=tokenizer_clf,
    clf_model=model_clf,
    retriever=retriever,
    faiss_index=index,
    corpus_texts=corpus_texts,
    nli_tokenizer=nli_tokenizer,
    nli_model=nli_model,
    device=DEVICE
)

print("🔮 FINAL DECISION")
print(output["final_prediction"])

print("\n🧠 NLI SCORES")
if "nli_scores" in output:
    print("\n🧠 NLI SCORES")
    print(output["nli_scores"])
else:
    print("\n🧠 NLI SCORES")
    print("Skipped (no reliable evidence)")


print("\n📚 TOP EVIDENCE")
for ev in output["retrieved_evidence"]:
    print("-", ev["text"])




DEBUG GATE: {'block': False, 'clarificatory': False, 'interpretive': False}
DEBUG INTERPRETIVE: False
🔮 FINAL DECISION
{'label': 'UNVERIFIABLE', 'reason': 'Authoritative claim without sufficient official confirmation', 'confidence': 1.0}

🧠 NLI SCORES

🧠 NLI SCORES
None

📚 TOP EVIDENCE
- Anonymous is a thing that exists..
- Anonymous represents many users.
- Anonymous represents the opposition of many online and offline community users.
- National Library of India is only the library of private record.
- Google has received significant criticism for privacy issues.


In [ ]:
print(route_claim(claim))
print(authoritative_gate(claim))


PIB
{'block': False, 'clarificatory': False, 'interpretive': False}
